V7 calendar policy: supply independent `/content/v7-calendar.json` containing ordered `trading_calendar`, nonblank `provenance`, per-date `expected_universe`, and nonblank `universe_provenance`. Price/source unions are audit only. INFO/WARN continue; numeric/OHLC errors quarantine observations. Default major outage is >=50% expected stocks missing/invalid; minimum usable training data is two dates with two targets in either head. Reports include versioned machine entries and Traditional Chinese summaries. Missing masked GroupD is allowed; active RS requires TWII.
Protocol v7-calendar-v2 rejects older prepared data/checkpoints. 60-calendar windows retain internal holes; exact raw-return horizons default to invalidating interior gaps. Per-head missing labels are allowed; no-target batches skip updates, whole no-update runs fail. Protected PIT fundamentals remain as-of; price rolling state restarts at gaps.
Builder runs only `python -B V6/experimental/v7_integrated_test.py` with explicit synthetic CPU SSD. Root supervisor independently runs CPU and official installed Mamba CUDA one-step (`--smoke-steps 1`), then refreshes delivery. No GPU success is claimed by CPU tests. Download/retry/source switching is next-stage work. See research/v7/integrated-candidate-colab.md for policy-aware prepare/data-check commands.


# MarketMamba V7 integrated candidate
Every cell is an explicit future Colab action. Synthetic/local evidence is not official Mamba-2, GPU, H1, performance, or promotion evidence.

## 0. Mount Drive and deliver reviewed source
This does not assume either Drive or `/content/MarketMamba` is already present. Put the reviewed repository snapshot at `MyDrive/MarketMamba-source`; it contains source only, not data, weights, or `.env`.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil, subprocess, sys
drive.mount('/content/drive')
SOURCE=Path('/content/drive/MyDrive/MarketMamba-source')
REPO=Path('/content/MarketMamba')
assert (SOURCE/'V6/experimental/v7_integrated_train.py').is_file(), SOURCE
if not REPO.exists(): shutil.copytree(SOURCE,REPO)
import hashlib
for rel in ['V6/experimental/v7_integrated_train.py','V6/experimental/v7_integrated_industry.py','V6/experimental/v7_integrated_config.py','V6/experimental/v7_integrated_prepare_cache.py']:
    assert (REPO/rel).is_file() and hashlib.sha256((REPO/rel).read_bytes()).digest()==hashlib.sha256((SOURCE/rel).read_bytes()).digest(), 'Local code differs from this delivery; start a fresh Colab runtime'
assert (REPO/'V6/experimental/v7_integrated_train.py').is_file(), REPO
DATA_ARCHIVE=Path('/content/drive/MyDrive/MarketMamba_V6/processed_v6.zip')
RAW=Path('/content/v7-input/processed_v6')
DATA=Path('/content/drive/MyDrive/MarketMamba_V7/prepared-resumable-20260911')
RUNTIME_DIR=Path('/content/v7-runtime'); RUNTIME_DIR.mkdir(parents=True,exist_ok=True)
MANIFEST=RUNTIME_DIR/'manifest.json'; RUNTIME=RUNTIME_DIR/'runtime.json'
TRAIN=REPO/'V6/experimental/v7_integrated_train.py'; PROBE=REPO/'V6/experimental/v7_integrated_probe.py'
if not MANIFEST.exists(): shutil.copy2(REPO/'V6/experimental/v7_integrated_environment.json',MANIFEST)


## 1. 安裝固定環境（首次執行）
將下格 INSTALL_LOCKED_ENVIRONMENT 設為 True，安裝完成後選 Restart session（不要刪除執行環境）。重新執行設定格，再將此開關設回 False。


In [ ]:
INSTALL_LOCKED_ENVIRONMENT=False
if INSTALL_LOCKED_ENVIRONMENT:
    subprocess.run([sys.executable,'-m','pip','install','--only-binary=:all:',
        '-r',str(REPO/'environments/v7-colab-2026.04/requirements.lock.txt')],check=True)
    print('Installation finished. Use Restart session, rerun setup, then leave this switch False.')


## 2. Optional known-compatible Torch runtime route
If the current Torch tuple has no pinned Mamba wheel, the official PyTorch previous-versions route below installs Torch 2.10.0 CUDA 12.8 binaries. It deliberately kills the kernel; reconnect and rerun cells 0–1 so metadata is recaptured from the new process.

In [ ]:
INSTALL_TORCH_210_CU128_AND_RESTART=False
if INSTALL_TORCH_210_CU128_AND_RESTART:
    import os, signal
    subprocess.run([sys.executable,'-m','pip','install','--only-binary=:all:',
        'torch==2.10.0','torchvision==0.25.0','torchaudio==2.10.0',
        '--index-url','https://download.pytorch.org/whl/cu128'],check=True)
    os.kill(os.getpid(),signal.SIGKILL)  # expected kernel restart


## 3. Resolve and install the exact official pinned Mamba wheel
The resolver maps CUDA 12.x metadata to `cu12`, distinguishes x86_64 from aarch64, requires exact v2.3.2.post1, and inspects versioned/applicable METADATA requirements. Resolve first, review, then install in a separate explicit run. No source build is permitted.

In [ ]:
RESOLVE_OFFICIAL_WHEEL=False; INSTALL_REVIEWED_WHEEL=False
resolve=[sys.executable,str(PROBE),'--setup-colab','--manifest',str(MANIFEST),'--runtime-metadata',str(RUNTIME),'--wheel-dir',str(RUNTIME_DIR)]
if RESOLVE_OFFICIAL_WHEEL: subprocess.run(resolve,check=True)
if INSTALL_REVIEWED_WHEEL: subprocess.run([*resolve,'--install'],check=True)


## 4. Prepare and check data
Restricted prices remain bounded while pre-window fundamental/revenue/dividend/cashflow and macro context is retained. Restricted outputs are always diagnostic-only. Stale TWII coverage blocks active RS features; inactive macro12 freshness does not.

In [ ]:
def run_logged(command, log_name):
    # The child can be stopped without deliberately restarting the notebook kernel.
    import os, signal, threading, time
    from collections import deque
    log_dir=DATA.parent/'prepare-logs'; log_dir.mkdir(parents=True,exist_ok=True)
    log_path=log_dir/log_name
    tail=deque(maxlen=35)
    with log_path.open('a',encoding='utf-8',buffering=1) as log:
        process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                                 text=True,bufsize=1,start_new_session=True)
        def stream_output():
            for line in process.stdout:
                log.write(line); log.flush()
                tail.append(line.rstrip())
                if line.startswith('{'):
                    print(line.rstrip(),flush=True)
        reader=threading.Thread(target=stream_output,daemon=True); reader.start()
        last=0; resource_stop=False
        try:
            while process.poll() is None:
                memory={}
                for line in Path('/proc/meminfo').read_text().splitlines():
                    key=line.partition(':')[0]
                    if key in ('MemAvailable','MemTotal'):
                        memory[key]=int(line.split()[1])
                if time.monotonic()-last>=30:
                    message=f"Preparing; available RAM={memory.get('MemAvailable',0)/1024**2:.1f} GiB; log={log_path}"
                    print(message,flush=True); log.write(message+'\n'); log.flush()
                    last=time.monotonic()
                if memory.get('MemAvailable',float('inf')) < max(2*1024**2, memory.get('MemTotal',0)*0.10):
                    resource_stop=True
                    os.killpg(process.pid,signal.SIGTERM)
                    break
                time.sleep(2)
            if resource_stop:
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    os.killpg(process.pid,signal.SIGKILL); process.wait()
            else:
                process.wait()
        except BaseException:
            if process.poll() is None:
                os.killpg(process.pid,signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    os.killpg(process.pid,signal.SIGKILL); process.wait()
            raise
        finally:
            reader.join(timeout=10)
        if process.returncode:
            print('\n'.join(tail))
            reason='Available RAM fell below reserve; completed stock checkpoints are preserved.' if resource_stop else f'Preparation child exited {process.returncode}.'
            raise RuntimeError(reason+' Full log: '+str(log_path))
    print('Completed:',log_path)


RUN_DIAGNOSTIC=False; RUN_FULL_PREPARE=False
# Supply independent calendars explicitly; never infer historical membership from prices or today's listings.
FULL_CALENDAR=REPO/'delivery/inputs/calendar-through-20260911.json'
SMOKE_CALENDAR=REPO/'delivery/inputs/smoke-calendar.json'  # supervisor's fixed seven-stock declared basket
FROZEN_DATE_TO='2026-09-11'
import json
if RUN_FULL_PREPARE or RUN_DIAGNOSTIC:
    import zipfile, tempfile, hashlib, json
    # The upload is a complete processed_v6 folder; no repair patch is applied.
    with zipfile.ZipFile(DATA_ARCHIVE) as archive:
        identity=hashlib.sha256(json.dumps([
            (item.filename,item.CRC,item.file_size) for item in archive.infolist()
        ]).encode()).hexdigest()
        marker=RAW/'.archive-identity'
        if RAW.exists():
            assert marker.is_file() and marker.read_text()==identity, 'Existing local data differs; use a fresh RAW directory'
        else:
            RAW.parent.mkdir(parents=True,exist_ok=True)
            with tempfile.TemporaryDirectory(dir=RAW.parent) as temporary:
                staging=Path(temporary)
                for item in archive.infolist():
                    destination=(staging/item.filename).resolve()
                    assert destination.is_relative_to(staging.resolve()), 'Unsafe archive path'
                    assert (item.external_attr >> 16) & 0o170000 != 0o120000, 'Archive contains symbolic links instead of data'
                archive.extractall(staging)
                required_files = ['prices_raw.parquet','revenue_raw.parquet','stock_info.parquet','macro_raw.parquet','knowledge_graph_v2.npz']
                candidates = sorted({
                    path.parent for path in staging.rglob('prices_raw.parquet')
                    if all((path.parent/name).is_file() for name in required_files)
                })
                if len(candidates) != 1:
                    entries = [str(path.relative_to(staging)) for path in staging.rglob('*') if path.is_file()][:30]
                    raise ValueError(
                        f'Expected one complete data folder, found {len(candidates)}. '
                        f'Required files: {required_files}. Archive entries (first 30): {entries}'
                    )
                source = candidates[0]
                print('Data folder found:', str(source.relative_to(staging)), flush=True)
                (source/'.archive-identity').write_text(identity)
                source.rename(RAW)
if RUN_DIAGNOSTIC:
    smoke=json.loads(SMOKE_CALENDAR.read_text())
    assert smoke['universe_provenance'].strip()
    smoke_ids=sorted({sid for ids in smoke['expected_universe'].values() for sid in ids})
    assert len(smoke_ids)==7 and smoke['trading_calendar'][-1] <= FROZEN_DATE_TO
    diagnostic=[sys.executable,str(TRAIN),'prepare','--calendar',str(SMOKE_CALENDAR),'--raw-dir',str(RAW),'--output-dir','/content/v7-diagnostic','--diagnostic','--stock-ids',*smoke_ids,'--date-from',smoke['trading_calendar'][0],'--date-to',smoke['trading_calendar'][-1]]
    run_logged(diagnostic, 'diagnostic-prepare.log')
prepare_full=[sys.executable,str(TRAIN),'prepare','--calendar',str(FULL_CALENDAR),'--raw-dir',str(RAW),'--output-dir',str(DATA),'--date-to',FROZEN_DATE_TO]
check=[sys.executable,str(TRAIN),'data-check','--feature-parquet',str(DATA/'features_59.parquet'),'--feature-metadata',str(DATA/'feature_metadata.json'),'--splits',str(DATA/'splits.json'),'--kg',str(DATA/'knowledge_graph_v2_csr.npz'),'--market-data',str(DATA/'market_prices_raw.parquet')]
if RUN_FULL_PREPARE:
    full=json.loads(FULL_CALENDAR.read_text())
    assert full['universe_provenance'].strip() and full['trading_calendar'][-1] == FROZEN_DATE_TO
    run_logged(prepare_full, 'full-prepare.log')
    run_logged(check, 'data-check.log')

if not RUN_FULL_PREPARE and not RUN_DIAGNOSTIC:
    print('資料準備尚未啟動：正式建置請將 RUN_FULL_PREPARE 設為 True。')


## 5. Guarded smoke, full train, and resume
Smoke has a separate checkpoint and cannot overwrite full training. Progress and exact resumable checkpoints are bounded to every 100 updates plus epoch/end/cap boundaries; `.best` is selected by validation only.

In [ ]:
RUN_SMOKE=False; RUN_FULL_TRAIN=False; RUN_RESUME=False
if RUN_SMOKE or RUN_FULL_TRAIN or RUN_RESUME:
    import json
    metadata=json.loads((DATA/'feature_metadata.json').read_text())
    assert metadata.get('industry_neutralization',{}).get('version')=='v7-industry-neutral-v1', 'Rebuild features using this industry-neutral delivery'
    assert metadata['industry_neutralization']['neutralized_cells']>0, 'No industry-neutral feature cells'
CKPT_DIR=Path('/content/drive/MyDrive/MarketMamba_V7/checkpoints-industry-20260911')
SMOKE_CKPT=CKPT_DIR/'v7-smoke.pt'; FULL_CKPT=CKPT_DIR/'v7-full.pt'
common=['--device','cuda','--runtime-metadata',str(RUNTIME),'--manifest',str(MANIFEST),'--feature-parquet',str(DATA/'features_59.parquet'),'--feature-metadata',str(DATA/'feature_metadata.json'),'--splits',str(DATA/'splits.json'),'--kg',str(DATA/'knowledge_graph_v2_csr.npz'),'--market-data',str(DATA/'market_prices_raw.parquet'),'--progress-interval','100','--checkpoint-interval','100']
smoke=[sys.executable,str(TRAIN),'smoke',*common,'--checkpoint',str(SMOKE_CKPT),'--smoke-steps','1']
full=[sys.executable,str(TRAIN),'train',*common,'--checkpoint',str(FULL_CKPT),'--epochs','20']
resume=[*full,'--resume',str(FULL_CKPT)]
if RUN_SMOKE: subprocess.run(smoke,check=True)
if RUN_FULL_TRAIN: subprocess.run(full,check=True)
if RUN_RESUME: subprocess.run(resume,check=True)


## 6. Forecast validation and optionally compare frozen baselines
Supply verified `v2_kg_nomacro` scores before comparison. The candidate forecast explicitly loads the validation-selected `.best` checkpoint; no comparison claim exists yet.

In [ ]:
RUN_FORECAST=False; RUN_EVALUATION=False
OUT=Path('/content/drive/MyDrive/MarketMamba_V7/evaluation-industry-20260911')
forecast=[sys.executable,str(TRAIN),'forecast',*common,'--split','validation','--checkpoint',str(FULL_CKPT)+'.best','--output-dir',str(OUT/'scores')]
evaluate=[sys.executable,str(TRAIN),'evaluate','--feature-parquet',str(DATA/'features_59.parquet'),'--feature-metadata',str(DATA/'feature_metadata.json'),'--splits',str(DATA/'splits.json'),'--kg',str(DATA/'knowledge_graph_v2_csr.npz'),'--candidate-scores',str(OUT/'scores/v7_integrated_scores.parquet'),'--baseline-scores','/content/drive/MyDrive/MarketMamba_V7/baselines/v2_kg_nomacro_scores.parquet','--market-data',str(DATA/'market_prices_raw.parquet'),'--output-dir',str(OUT/'comparison')]
if RUN_FORECAST: subprocess.run(forecast,check=True)
if RUN_EVALUATION: subprocess.run(evaluate,check=True)
